# Kết nối Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

folder = "/content/drive/MyDrive/IE224-DataAnalysis"

Mounted at /content/drive/


# Kết nối Github


In [ ]:
!git clone https://github.com/TTTThanh2812/IE224.O11.CNCL-Data_Analysis

Cloning into 'IE224.O11.CNCL-Data_Analysis'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 30 (delta 4), reused 0 (delta 0), pack-reused 0
Receiving objects: 100% (30/30), 291.44 KiB | 3.74 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [ ]:
data_path = '/content/IE224.O11.CNCL-Data_Analysis/data/final_data.csv'

# Import thư viện

In [ ]:
from tqdm import tqdm
from tabulate import tabulate
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

# Load dataset

In [ ]:
df = pd.read_csv(data_path)
df

,pc_brand,pc_model,elt_condition,elt_warranty,desktop_screen_size,pc_cpu,pc_ram,pc_vga,pc_drive_capacity,elt_origin,usage_information,classify,price
0,NaN,NaN,Đã sử dụng (chưa sửa chữa),>12 tháng,>= 21 inch,Intel Xeon,> 32GB,NVIDIA,256 GB,Việt Nam,In trên bao bì,desktop,13.500.000 đ
1,NaN,NaN,Đã sử dụng (chưa sửa chữa),Còn bảo hành,>= 21 inch,Intel Core i5,8 GB,Khác,500 GB,Đang cập nhật,In trên bao bì,desktop,1.800.000 đ
2,NaN,NaN,Đã sử dụng (chưa sửa chữa),Còn bảo hành,>= 21 inch,Intel Xeon,32 GB,NVIDIA,256 GB,Đang cập nhật,In trên bao bì,desktop,8.500.000 đ
3,NaN,NaN,Mới,Còn bảo hành,>= 21 inch,Intel Xeon,> 32GB,NVIDIA,256 GB,Việt Nam,In trên bao bì,desktop,12.500.000 đ
4,NaN,NaN,Đã sử dụng (chưa sửa chữa),3 tháng,Không bán kèm màn hình,Intel Core i3,8 GB,NVIDIA,128 GB,Đang cập nhật,In trên bao bì,desktop,6.300.000 đ
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999,HP,ProBook,Đã sử dụng (chưa sửa chữa),Bảo hành hãng,NaN,Intel Core i5,8 GB,NaN,256 GB,Đang cập nhật,In trên bao bì,laptop,5.200.000 đ
8000,Apple,MacBook Air,Đã sử dụng (chưa sửa chữa),Bảo hành hãng,NaN,NaN,NaN,NaN,NaN,Đang cập nhật,In trên bao bì,laptop,3.990.000 đ
8001,Dell,Latitude,Đã sử dụng (chưa sửa chữa),3 tháng,13 - 14.9 inch,Intel Core i7,8 GB,NVIDIA,256 GB,Mỹ,In trên bao bì,laptop,7.500.000 đ
8002,Dell,Vostro,Đã sử dụng (chưa sửa chữa),1 tháng,13 - 14.9 inch,Intel Core i5,4 GB,Onboard,1 TB,Đang cập nhật,In trên bao bì,laptop,3.600.000 đ


# EDA prev preprocessing

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8004 entries, 0 to 8003
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   pc_brand             3994 non-null   object
 1   pc_model             3994 non-null   object
 2   elt_condition        7889 non-null   object
 3   elt_warranty         7889 non-null   object
 4   desktop_screen_size  5889 non-null   object
 5   pc_cpu               6761 non-null   object
 6   pc_ram               6727 non-null   object
 7   pc_vga               5639 non-null   object
 8   pc_drive_capacity    6555 non-null   object
 9   elt_origin           7889 non-null   object
 10  usage_information    7889 non-null   object
 11  classify             8004 non-null   object
 12  price                7889 non-null   object
dtypes: object(13)
memory usage: 813.0+ KB


In [ ]:
df.duplicated().any()

True

In [ ]:
def table_unique (df):
  chunk_size = 4
  unique_values = []

  for column in df.columns:
    if column in ["price"]:
      continue
    buffer = [item for item in map(str, df[column].unique())]
    unique_values.append({
      'name': column,
      'values':',\n'.join([', '.join(buffer[i:i+chunk_size])
      for i in range(0, len(buffer), chunk_size)]),
      'number_of_unique': len(buffer)})

  print(tabulate(
    unique_values,
    headers={"name": "Column Name",
            "values": "Unique Values",
            "number_of_unique": "Number of Unique values"},
    tablefmt="grid"))


In [ ]:
table_unique(df)

+---------------------+---------------------------------------------------------------------------+---------------------------+
| Column Name         | Unique Values                                                             |   Number of Unique values |
+=====================+===========================================================================+===========================+
| pc_brand            | nan, Asus, HP, Dell,                                                      |                        16 |
|                     | Apple, Sony, Toshiba, Lenovo,                                             |                           |
|                     | Microsoft, Panasonic, Hãng Khác, MSI,                                     |                           |
|                     | Acer, Samsung, Razer, LG                                                  |                           |
+---------------------+---------------------------------------------------------------------------+-----

# Preprocessing

## Xóa dataset trùng nhau

In [ ]:
df_prep = df.drop_duplicates()

In [ ]:
df_prep.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7356 entries, 0 to 8003
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   pc_brand             3885 non-null   object
 1   pc_model             3885 non-null   object
 2   elt_condition        7355 non-null   object
 3   elt_warranty         7355 non-null   object
 4   desktop_screen_size  5650 non-null   object
 5   pc_cpu               6468 non-null   object
 6   pc_ram               6439 non-null   object
 7   pc_vga               5434 non-null   object
 8   pc_drive_capacity    6279 non-null   object
 9   elt_origin           7355 non-null   object
 10  usage_information    7355 non-null   object
 11  classify             7356 non-null   object
 12  price                7355 non-null   object
dtypes: object(13)
memory usage: 804.6+ KB


## Xóa các hàng có giá trị price, pc_cpu, pc_ram, pc_drive_capacity là nan

In [ ]:
df_prep = df_prep.dropna(subset=['price'])

In [ ]:
df_prep.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7355 entries, 0 to 8003
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   pc_brand             3885 non-null   object
 1   pc_model             3885 non-null   object
 2   elt_condition        7355 non-null   object
 3   elt_warranty         7355 non-null   object
 4   desktop_screen_size  5650 non-null   object
 5   pc_cpu               6468 non-null   object
 6   pc_ram               6439 non-null   object
 7   pc_vga               5434 non-null   object
 8   pc_drive_capacity    6279 non-null   object
 9   elt_origin           7355 non-null   object
 10  usage_information    7355 non-null   object
 11  classify             7355 non-null   object
 12  price                7355 non-null   object
dtypes: object(13)
memory usage: 804.5+ KB


In [ ]:
df_prep = df_prep.dropna(subset=['pc_cpu', 'pc_ram', 'pc_drive_capacity'], how='all')

In [ ]:
df_prep.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 6636 entries, 0 to 8003
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   pc_brand             3419 non-null   object
 1   pc_model             3419 non-null   object
 2   elt_condition        6636 non-null   object
 3   elt_warranty         6636 non-null   object
 4   desktop_screen_size  5620 non-null   object
 5   pc_cpu               6468 non-null   object
 6   pc_ram               6439 non-null   object
 7   pc_vga               5423 non-null   object
 8   pc_drive_capacity    6279 non-null   object
 9   elt_origin           6636 non-null   object
 10  usage_information    6636 non-null   object
 11  classify             6636 non-null   object
 12  price                6636 non-null   object
dtypes: object(13)
memory usage: 725.8+ KB


## Ghi đè các giá trị nhiễu trong dataset

In [ ]:
df_prep.isnull().sum()

pc_brand               3217
pc_model               3217
elt_condition             0
elt_warranty              0
desktop_screen_size    1016
pc_cpu                  168
pc_ram                  197
pc_vga                 1213
pc_drive_capacity       357
elt_origin                0
usage_information         0
classify                  0
price                     0
dtype: int64

### pc_vga

In [ ]:
df_prep['pc_vga'].value_counts(normalize=True).idxmax()

'Onboard'

In [ ]:
df_prep['pc_vga'] = df_prep['pc_vga'].replace(['Khác'], 'Onboard')
df_prep['pc_vga'] = df_prep['pc_vga'].fillna('Onboard')

### elt_warranty

In [ ]:
# df_prep['elt_warranty'] = df_prep['elt_warranty'].replace(
#   {'Bảo hành hãng': '0 tháng',
#     'Hết bảo hành': '0 tháng',
#     'Còn bảo hành': '1 tháng',
#     '4-6 tháng': '5 tháng',
#     '7-12 tháng': '9 tháng',
#     '>12 tháng': '12 tháng'}
# )

In [ ]:
# df_prep['elt_warranty'] = df_prep['elt_warranty'].str.replace(" tháng",'')
# df_prep['elt_warranty'] = df_prep['elt_warranty'].astype('int64')
# df_prep.rename(columns = {'elt_warranty':'elt_warranty (tháng)'}, inplace = True)

In [ ]:
df_prep['elt_warranty'] = df_prep['elt_warranty'].replace(['Bảo hành hãng'], 'Hết bảo hành')

### elt_origin

In [ ]:
df_prep['elt_origin'] = df_prep['elt_origin'].replace(['Đang cập nhật'], 'Nước khác')

### desktop_screen_size

In [ ]:
max_desktop_screen_size_laptop = df_prep[df_prep['classify'] == 'laptop']['desktop_screen_size'].value_counts(normalize=True).idxmax()
print("Desktop",df_prep[df_prep['classify'] == 'desktop']['desktop_screen_size'].value_counts(normalize=True))

Desktop >= 21 inch                0.463597
Không bán kèm màn hình    0.337662
19 - 20.9 inch            0.137741
17 - 18.9 inch            0.039748
15 - 16.9 inch            0.021251
Name: desktop_screen_size, dtype: float64


In [ ]:
df_prep.loc[df_prep['classify'] == 'laptop', 'desktop_screen_size'] = df_prep.loc[df_prep['classify'] == 'laptop', 'desktop_screen_size'].fillna(max_desktop_screen_size_laptop)
df_prep.loc[df_prep['classify'] == 'desktop', 'desktop_screen_size'] = df_prep.loc[df_prep['classify'] == 'desktop', 'desktop_screen_size'].fillna('Không bán kèm màn hình')

In [ ]:
# df_prep['desktop_screen_size'] = df_prep['desktop_screen_size'].replace(
#   {'>= 21 inch': '21 inch',
#    'Không bán kèm màn hình': '0 inch',
#    '19 - 20.9 inch': '20 inch',
#    '15 - 16.9 inch': '16 inch',
#    '17 - 18.9 inch': '18 inch',
#    '13 - 14.9 inch': '14 inch',
#    '11 - 12.9 inch': '12 inch',
#    '9 - 10.9 inch': '10 inch',
#    '< 9 inch': '9 inch'}
# )

In [ ]:
# df_prep['desktop_screen_size'] = df_prep['desktop_screen_size'].str.replace(" inch",'')
# df_prep['desktop_screen_size'] = df_prep['desktop_screen_size'].astype('int64')
# df_prep.rename(columns = {'desktop_screen_size':'desktop_screen_size (inch)'}, inplace = True)

### pc_ram

In [ ]:
max_pc_ram_laptop = df_prep[df_prep['classify'] == 'laptop']['pc_ram'].value_counts(normalize=True).idxmax()
max_pc_ram_desktop = df_prep[df_prep['classify'] == 'desktop']['pc_ram'].value_counts(normalize=True).idxmax()

In [ ]:
df_prep.loc[df_prep['classify'] == 'laptop', 'pc_ram'] = df_prep.loc[df_prep['classify'] == 'laptop', 'pc_ram'].fillna(max_pc_ram_laptop)
df_prep.loc[df_prep['classify'] == 'desktop', 'pc_ram'] = df_prep.loc[df_prep['classify'] == 'desktop', 'pc_ram'].fillna(max_pc_ram_desktop)

In [ ]:
# df_prep['pc_ram'] = df_prep['pc_ram'].replace(
#   {'> 32GB': '64 GB',
#    '< 1 GB': '0.5 GB',}
# )

In [ ]:
# df_prep['pc_ram'] = df_prep['pc_ram'].str.replace(" GB",'')
# df_prep['pc_ram'] = df_prep['pc_ram'].astype('float64')
# df_prep.rename(columns = {'pc_ram':'pc_ram (GB)'}, inplace = True)

### pc_drive_capacity

In [ ]:
max_pc_drive_capacity_laptop = df_prep[df_prep['classify'] == 'laptop']['pc_drive_capacity'].value_counts(normalize=True).idxmax()
max_pc_drive_capacity_desktop = df_prep[df_prep['classify'] == 'desktop']['pc_drive_capacity'].value_counts(normalize=True).idxmax()

In [ ]:
df_prep.loc[df_prep['classify'] == 'laptop', 'pc_drive_capacity'] = df_prep.loc[df_prep['classify'] == 'laptop', 'pc_drive_capacity'].fillna(max_pc_drive_capacity_laptop)
df_prep.loc[df_prep['classify'] == 'desktop', 'pc_drive_capacity'] = df_prep.loc[df_prep['classify'] == 'desktop', 'pc_drive_capacity'].fillna(max_pc_drive_capacity_desktop)

In [ ]:
# df_prep['pc_drive_capacity'] = df_prep['pc_drive_capacity'].replace(
#   {'< 128 GB': '120 GB',
#    '> 1 TB': '1500 GB',
#    '1 TB': '1000 GB',}
# )

In [ ]:
# df_prep['pc_drive_capacity'] = df_prep['pc_drive_capacity'].str.replace(" GB",'')
# df_prep['pc_drive_capacity'] = df_prep['pc_drive_capacity'].astype('int64')
# df_prep.rename(columns = {'pc_drive_capacity':'pc_drive_capacity (GB)'}, inplace = True)

### pc_model

In [ ]:
df_prep.loc[df_prep['classify'] == 'desktop', ['pc_brand', 'pc_model']] = 'desktop'
df_prep['pc_model'] = df_prep['pc_model'].replace(['Dòng Khác'], 'Dòng khác')

df_pc_brand = df_prep['pc_brand'].value_counts().index
brand_model_max_dict = {}

for brand in df_pc_brand:
  model_counts = df_prep[df_prep['pc_brand'] == brand]['pc_model'].value_counts(normalize=True)
  brand_model_max_dict[brand] = model_counts.idxmax()
  if brand != "desktop" and brand != "Hãng Khác" and len(model_counts) >= 2:
    if brand_model_max_dict[brand] == "Dòng khác":
      brand_model_max_dict[brand] = model_counts.nlargest(2).index[1]

for brand, max_model in brand_model_max_dict.items():
  df_prep.loc[df_prep['pc_brand'] == brand, 'pc_model'] = df_prep.loc[df_prep['pc_brand'] == brand, 'pc_model'].replace(['Dòng khác'], max_model)

### pc_cpu

In [ ]:
df_pc_model = df_prep['pc_model'].value_counts().index
model_cpu_max_dict = {}

for model in df_pc_model:
  df_prep.loc[df_prep['pc_model'] == model, 'pc_cpu'] = df_prep.loc[df_prep['pc_model'] == model, 'pc_cpu'].fillna('Khác')

for model in df_pc_model:
  cpu_counts = df_prep[df_prep['pc_model'] == model]['pc_cpu'].value_counts(normalize=True)
  model_cpu_max_dict[model] = cpu_counts.idxmax()
  if len(cpu_counts) >= 2:
    if cpu_counts.idxmax() == "Khác":
      if model != "MacBook Air M1" and  model != "MacBook Pro M1 Touch Bar" and model != "Surface Pro X":
        model_cpu_max_dict[model] = cpu_counts.nlargest(2).index[1]
  if model == "MacBook Air M1" or model == "MacBook Pro M1" or model == "MacBook Pro M1 Touch Bar":
    model_cpu_max_dict[model] = "Apple M1"
  elif model == "MacBook Air M2" or model == "MacBook Pro M2" or model == "MacBook Pro M2 Touch Bar":
    model_cpu_max_dict[model] = "Apple M2"
  elif model == "Surface Pro X":
    model_cpu_max_dict[model] = "Microsoft SQ2"
  elif model == "Notebook 7":
    model_cpu_max_dict[model] = "Intel Core i5"

for model, max_cpu in model_cpu_max_dict.items():
  df_prep.loc[df_prep['pc_model'] == model, 'pc_cpu'] = df_prep.loc[df_prep['pc_model'] == model, 'pc_cpu'].replace(['Khác'], max_cpu)

### price

In [ ]:
df_prep['price'] = df_prep['price'].str.replace(" đ",'').str.replace(".",'')
df_prep['price'] = df_prep['price'].astype('float64')
df_prep.rename(columns = {'price':'price (VND)'}, inplace = True)

<ipython-input-28-c0a48485c21f>:1: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df_prep['price'] = df_prep['price'].str.replace(" đ",'').str.replace(".",'')


### usage_information

In [ ]:
df_prep = df_prep.drop(columns=['usage_information'])

### List

In [ ]:
df_prep = df_prep.reset_index(drop=True)
df_prep

,pc_brand,pc_model,elt_condition,elt_warranty,desktop_screen_size,pc_cpu,pc_ram,pc_vga,pc_drive_capacity,elt_origin,classify,price (VND)
0,desktop,desktop,Đã sử dụng (chưa sửa chữa),>12 tháng,>= 21 inch,Intel Xeon,> 32GB,NVIDIA,256 GB,Việt Nam,desktop,13500000.0
1,desktop,desktop,Đã sử dụng (chưa sửa chữa),Còn bảo hành,>= 21 inch,Intel Core i5,8 GB,Onboard,500 GB,Nước khác,desktop,1800000.0
2,desktop,desktop,Đã sử dụng (chưa sửa chữa),Còn bảo hành,>= 21 inch,Intel Xeon,32 GB,NVIDIA,256 GB,Nước khác,desktop,8500000.0
3,desktop,desktop,Mới,Còn bảo hành,>= 21 inch,Intel Xeon,> 32GB,NVIDIA,256 GB,Việt Nam,desktop,12500000.0
4,desktop,desktop,Đã sử dụng (chưa sửa chữa),3 tháng,Không bán kèm màn hình,Intel Core i3,8 GB,NVIDIA,128 GB,Nước khác,desktop,6300000.0
...,...,...,...,...,...,...,...,...,...,...,...,...
6631,Microsoft,Surface Pro X,Đã sử dụng (chưa sửa chữa),Hết bảo hành,13 - 14.9 inch,Intel Core i5,8 GB,Onboard,256 GB,Nước khác,laptop,8900000.0
6632,HP,ProBook,Đã sử dụng (chưa sửa chữa),Hết bảo hành,13 - 14.9 inch,Intel Core i5,8 GB,Onboard,256 GB,Nước khác,laptop,5200000.0
6633,Dell,Latitude,Đã sử dụng (chưa sửa chữa),3 tháng,13 - 14.9 inch,Intel Core i7,8 GB,NVIDIA,256 GB,Mỹ,laptop,7500000.0
6634,Dell,Vostro,Đã sử dụng (chưa sửa chữa),1 tháng,13 - 14.9 inch,Intel Core i5,4 GB,Onboard,1 TB,Nước khác,laptop,3600000.0


### pc_cpu_label


In [ ]:
def get_cpu_label(cpu):
  if 'Intel' in cpu:
    return 'Intel'
  elif 'AMD' in cpu or 'Ryzen' in cpu or 'Athlon' in cpu:
    return 'AMD'
  elif 'Apple' in cpu:
    return 'Apple'
  elif 'Microsoft' in cpu:
    return 'Microsoft'
  else:
    return 'Other'

In [ ]:
df_prep['pc_cpu_label'] = df_prep['pc_cpu'].apply(get_cpu_label)

# Label encoder

In [ ]:
def encode(df, column):
  if df[column].dtype == 'object':
    df[column] = df[column].astype('category')

    le = LabelEncoder()
    df[column] = le.fit_transform(df[column])
  return df

In [ ]:
df_prep_onehot = df_prep.copy()

In [ ]:
df_columns = df_prep_onehot.select_dtypes(include='object').columns

In [ ]:
for column in df_columns:
  df_prep_onehot = encode(df_prep_onehot, column)

In [ ]:
df_prep_onehot

,pc_brand,pc_model,elt_condition,elt_warranty,desktop_screen_size,pc_cpu,pc_ram,pc_vga,pc_drive_capacity,elt_origin,classify,price (VND),pc_cpu_label
0,15,101,1,5,7,14,8,1,3,6,0,13500000.0,2
1,15,101,1,6,7,9,6,2,6,3,0,1800000.0,2
2,15,101,1,6,7,14,3,1,3,3,0,8500000.0,2
3,15,101,0,6,7,14,8,1,3,6,0,12500000.0,2
4,15,101,1,2,8,8,6,1,1,3,0,6300000.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6631,9,83,1,7,1,9,6,2,3,3,1,8900000.0,2
6632,4,60,1,7,1,9,6,2,3,3,1,5200000.0,2
6633,3,36,1,2,1,10,6,1,3,1,1,7500000.0,2
6634,3,95,1,0,1,9,4,2,0,3,1,3600000.0,2


In [ ]:
df_prep_onehot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6636 entries, 0 to 6635
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   pc_brand             6636 non-null   int64  
 1   pc_model             6636 non-null   int64  
 2   elt_condition        6636 non-null   int64  
 3   elt_warranty         6636 non-null   int64  
 4   desktop_screen_size  6636 non-null   int64  
 5   pc_cpu               6636 non-null   int64  
 6   pc_ram               6636 non-null   int64  
 7   pc_vga               6636 non-null   int64  
 8   pc_drive_capacity    6636 non-null   int64  
 9   elt_origin           6636 non-null   int64  
 10  classify             6636 non-null   int64  
 11  price (VND)          6636 non-null   float64
 12  pc_cpu_label         6636 non-null   int64  
dtypes: float64(1), int64(12)
memory usage: 674.1 KB


# Lưu dataset preprocessing

## df prep v2

In [ ]:
df_prep.to_csv(folder + '/data_preprocessing.csv')

In [ ]:
df_prep_onehot.to_csv(folder + '/data_preprocessing_labelencoder.csv')